### Data Ingetion ###

In [ ]:
### Document Structure

from langchain_core.documents import Document

c:\Users\Dell\Desktop\RAG\rag\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [ ]:
doc = Document(

    page_content="this is a page content to create RAG", 
    metadata= {
        "source": "example.txt",
        "author": "Bade Sahab",
        "pages": 1,
        "date_creation": "2026-03-22"

    }
)
doc

Document(metadata={'source': 'example.txt', 'author': 'Bade Sahab', 'pages': 1, 'date_creation': '2026-03-22'}, page_content='this is a page content to create RAG')

In [ ]:
### Create a txt file
import os
os.makedirs("../data/text_files",exist_ok=True)


In [ ]:
sample_texts={
    "../data/text_files/python_intro.txt":""" 

The Server-Sent Events (SSE) API enables pushing messages/updates from a server to the web page via HTTP connection.

Server-Sent Events - One Way Messaging
A server-sent event is when a web page automatically gets messages/updates from a server.

Normally, a web page has to request data from the server, but with server-sent events, the updates are pushed automatically.

Examples: Facebook/Twitter updates, stock market updates, news feeds, sport results, etc.

Browser Support
The numbers in the table specify the first browser version that fully support the Server-Sent Events API.

API					
SSE	6.0	79.0	6.0	5.0	11.5
Receive Server-Sent Event Notifications
The EventSource object is used to receive server-sent event notifications:

Example
<script>
const x = document.getElementById("result");
// Check browser support for SSE
if(typeof(EventSource) !== "undefined") {
  var source = new EventSource("demo_sse.php");
  source.onmessage = function(event) {
    x.innerHTML += event.data + "<br>";
  };
} else {
  x.innerHTML = "Sorry, no support for server-sent events.";
}
</script>
Example explained:

Create a new EventSource object, and specify the URL of the page sending the updates (in this example "demo_sse.php")
Each time an update is received, the onmessage event occurs
When an onmessage event occurs, put the received data into the element with id="result"


"""
}

for filepath,content in sample_texts.items():
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content)
print("Sample text file created")

Sample text file created


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding = "utf-8")
document = loader.load()
print(document)

c:\Users\Dell\Desktop\RAG\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content=' \n\nThe Server-Sent Events (SSE) API enables pushing messages/updates from a server to the web page via HTTP connection.\n\nServer-Sent Events - One Way Messaging\nA server-sent event is when a web page automatically gets messages/updates from a server.\n\nNormally, a web page has to request data from the server, but with server-sent events, the updates are pushed automatically.\n\nExamples: Facebook/Twitter updates, stock market updates, news feeds, sport results, etc.\n\nBrowser Support\nThe numbers in the table specify the first browser version that fully support the Server-Sent Events API.\n\nAPI\t\t\t\t\t\nSSE\t6.0\t79.0\t6.0\t5.0\t11.5\nReceive Server-Sent Event Notifications\nThe EventSource object is used to receive server-sent event notifications:\n\nExample\n<script>\nconst x = document.getElementById("result");\n// Check browser support for SSE\nif(typeof(EventSource) !== "undefined") {\n  v

In [ ]:
### Directory loader
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyMuPDFLoader, DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls= PyMuPDFLoader,
    show_progress=False
)

pdf_documents = dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Mac OS X 10.11.6 Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20180605201708Z00'00'", 'source': '..\\data\\pdf\\Company-Policy-and-Procedure-June-1.18-V6.0.pdf', 'file_path': '..\\data\\pdf\\Company-Policy-and-Procedure-June-1.18-V6.0.pdf', 'total_pages': 109, 'format': 'PDF 1.4', 'title': 'Microsoft Word - Company Policy and Procedure June 1.18 V6.0.docx', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20180605201708Z00'00'", 'trapped': '', 'modDate': "D:20180605201708Z00'00'", 'creationDate': "D:20180605201708Z00'00'", 'page': 0}, page_content='Company Policy \nand Procedure \nManual \nTriageLogic, LLC \nInitial Version November 2013 \nLast update June 4, 2018 \nVersion 6.0 \n \n \nApproved By: \n \n \n \nCharu G. Raheja, PhD \nChair/CEO \nTriageLogic, LL'),
 Document(metadata={'producer': 'Mac OS X 10.11.6 Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20180605201708Z00'00'", 'source': '..\\data\\pdf\\Company-Polic

In [ ]:
type(pdf_documents[0])

langchain_core.documents.base.Document

In [ ]:
# Creating Data Chunks 

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [ ]:
chunks=split_documents(pdf_documents)
chunks

Split 132 documents into 315 chunks

Example chunk:
Content: Company Policy 
and Procedure 
Manual 
TriageLogic, LLC 
Initial Version November 2013 
Last update June 4, 2018 
Version 6.0 
 
 
Approved By: 
 
 
 
Charu G. Raheja, PhD 
Chair/CEO 
TriageLogic, LL...
Metadata: {'producer': 'Mac OS X 10.11.6 Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20180605201708Z00'00'", 'source': '..\\data\\pdf\\Company-Policy-and-Procedure-June-1.18-V6.0.pdf', 'file_path': '..\\data\\pdf\\Company-Policy-and-Procedure-June-1.18-V6.0.pdf', 'total_pages': 109, 'format': 'PDF 1.4', 'title': 'Microsoft Word - Company Policy and Procedure June 1.18 V6.0.docx', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20180605201708Z00'00'", 'trapped': '', 'modDate': "D:20180605201708Z00'00'", 'creationDate': "D:20180605201708Z00'00'", 'page': 0}


[Document(metadata={'producer': 'Mac OS X 10.11.6 Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20180605201708Z00'00'", 'source': '..\\data\\pdf\\Company-Policy-and-Procedure-June-1.18-V6.0.pdf', 'file_path': '..\\data\\pdf\\Company-Policy-and-Procedure-June-1.18-V6.0.pdf', 'total_pages': 109, 'format': 'PDF 1.4', 'title': 'Microsoft Word - Company Policy and Procedure June 1.18 V6.0.docx', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20180605201708Z00'00'", 'trapped': '', 'modDate': "D:20180605201708Z00'00'", 'creationDate': "D:20180605201708Z00'00'", 'page': 0}, page_content='Company Policy \nand Procedure \nManual \nTriageLogic, LLC \nInitial Version November 2013 \nLast update June 4, 2018 \nVersion 6.0 \n \n \nApproved By: \n \n \n \nCharu G. Raheja, PhD \nChair/CEO \nTriageLogic, LL'),
 Document(metadata={'producer': 'Mac OS X 10.11.6 Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20180605201708Z00'00'", 'source': '..\\data\\pdf\\Company-Polic

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer 
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
              model_name: HuggingFace model name for sentense embedding 
        """ 

        self.model_name = model_name
        self.model = None
        self._load_model()

    
    def _load_model(self):
        "Load sentenceTransformer model"
        try:
            print(f'Loading embedding model: {self.model_name}')
            self.model = SentenceTransformer(self.model_name)
            print(f'Model loaded successfully. Embedding dimention: {self.model.get_sentence_embedding_dimension()}')
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embedding for a list of text

        Args:
        numpy array of embedding with shape (len(texts), embedding_dim)

        """

        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generate embedding for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f'Generated embeddings with shape: {embeddings.shape}')
        return embeddings
    
    def get_embedding_dimention(self) -> int:
        """Get the embedding dimention of the model"""

        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()
    

## Initialize the embedding manager

Embedding_Manager = EmbeddingManager()

Embedding_Manager

              
              



Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4082.49it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimention: 384


### Vector Store

In [ ]:
import os
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 428


In [ ]:
chunks

[Document(metadata={'producer': 'Mac OS X 10.11.6 Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20180605201708Z00'00'", 'source': '..\\data\\pdf\\Company-Policy-and-Procedure-June-1.18-V6.0.pdf', 'file_path': '..\\data\\pdf\\Company-Policy-and-Procedure-June-1.18-V6.0.pdf', 'total_pages': 109, 'format': 'PDF 1.4', 'title': 'Microsoft Word - Company Policy and Procedure June 1.18 V6.0.docx', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20180605201708Z00'00'", 'trapped': '', 'modDate': "D:20180605201708Z00'00'", 'creationDate': "D:20180605201708Z00'00'", 'page': 0}, page_content='Company Policy \nand Procedure \nManual \nTriageLogic, LLC \nInitial Version November 2013 \nLast update June 4, 2018 \nVersion 6.0 \n \n \nApproved By: \n \n \n \nCharu G. Raheja, PhD \nChair/CEO \nTriageLogic, LL'),
 Document(metadata={'producer': 'Mac OS X 10.11.6 Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20180605201708Z00'00'", 'source': '..\\data\\pdf\\Company-Polic

In [ ]:
### convert text to embeddings 
text = [doc.page_content for doc in chunks]

### Generate Embaddings
embedding = Embedding_Manager.generate_embeddings(text)

### store in the vector DB
vectorstore.add_documents(chunks, embedding)

Generate embedding for 315 texts...


Batches: 100%|██████████| 10/10 [00:11<00:00,  1.18s/it]


Generated embeddings with shape: (315, 384)
Adding 315 documents to vector store...
Successfully added 315 documents to vector store
Total documents in collection: 743


## Retriever Pipeline From VectorStore 

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever 

        Args:
        vector_store: Vector store containing documents embeddings
        embedding_manager: Manager for generating query embedding
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,Embedding_Manager)

In [ ]:
rag_retriever.retrieve("Workmen’s Compensation Insurance Policy ")

Retrieving documents for query: 'Workmen’s Compensation Insurance Policy '
Top K: 5, Score threshold: 0.0
Generate embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 76.33it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_11e42fe9_294',
  'content': 'IdeasInc. Management (P) Ltd.   \nSoftware Development                            HR Outsourcing                         Training                          Security Services \n \n \n \n \n \n\uf0fc Workmen’s Compensation Insurance Policy for the employees deployed in \nNon-implement area of ESIC are obtained separately. \n\uf0fc Labour License under Contract Labour (regulation of Employment and \nAbolition) Act is obtained separately for all units having deployment of 20 or \nmore employees. \n\uf0fc ESIC Registration',
  'metadata': {'creationDate': "D:20130308185444+05'30'",
   'modDate': "D:20130308185444+05'30'",
   'file_path': '..\\data\\pdf\\Ideasinc.pdf',
   'content_length': 495,
   'source': '..\\data\\pdf\\Ideasinc.pdf',
   'keywords': '',
   'creator': 'Microsoft® Office Word 2007',
   'format': 'PDF 1.5',
   'doc_index': 294,
   'title': 'COMPANY PROFILE',
   'author': 'kashyap',
   'total_pages': 23,
   'subject': 'IDEAS INC MANAGE

### Integration Vectordb Context pipeline with LLM output

In [ ]:
from langchain_ollama import ChatOllama
import os
from dotenv import load_dotenv

load_dotenv("../.env")

llm = ChatOllama(model="phi3", base_url="http://localhost:11434")

def rag_simple(query, retriever, llm, top_k=3):
    result = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join([doc['content'] for doc in result]) if result else ""
    
    if not context:
        return "No relevant context found"
    
    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question:
{query}

Answer:"""
    
    response = llm.invoke(prompt)
    return response.content

In [ ]:
answer = rag_simple("""What are Law and Regulatory Compliance – General? """, rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What are Law and Regulatory Compliance – General? '
Top K: 3, Score threshold: 0.0
Generate embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 51.38it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Law and Regulatory Compliance – General refers to TriageLogic's voluntary implementation of a compliance program designed to foster an environment of ethical and legal behavior in support of its mission.


Explain Document Management Policy in detail

### Enhancement

In [ ]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output



In [ ]:
result = rag_advanced("""What are Law and Regulatory Compliance – General?""", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What are Law and Regulatory Compliance – General?'
Top K: 3, Score threshold: 0.1
Generate embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 62.57it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Answer: Law and Regulatory Compliance – General involves reporting and communicating changes to new laws, regulations, and policies to affected employees, and holding all members, including supervisors, accountable for complying with applicable standards, laws, and procedures, with disciplinary actions for non-compliance.
Sources: [{'source': '..\\data\\pdf\\Company-Policy-and-Procedure-June-1.18-V6.0.pdf', 'page': 23, 'score': 0.12994515895843506, 'preview': 'Compliance officer or the CEO will report any changes to policy and new laws and regulations \nduring the general management meeting. Such matters will be documented in the minutes and \nemailed to all employees who are affected by the new law or policy. \n \n5. \nDisciplinary Guidelines \n \nAll members o...'}, {'source': '..\\data\\pdf\\Company-Policy-and-Procedure-June-1.18-V6.0.pdf', 'page': 23, 'score': 0.12994515895843506, 'preview': 'Compliance officer or the CEO will report any changes to policy and new laws and regulatio